In [20]:
import pandas as pd
df = pd.read_csv("superstore.csv", encoding="latin1")
df.head(5)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [21]:
df.groupby("Region")["Sales"].sum() # Understanding GroupBy & Finding Total Sales by Region

Region
Central    501239.8908
East       678781.2400
South      391721.9050
West       725457.8245
Name: Sales, dtype: float64

In [22]:
df.groupby(["Region", "Category"])[["Sales", "Profit"]].sum() # Grouping by multiple columns and aggregating multiple columns

Sales      Profit
Region  Category                                
Central Furniture        163797.1638  -2871.0494
        Office Supplies  167026.4150   8879.9799
        Technology       170416.3120  33697.4320
East    Furniture        208291.2040   3046.1658
        Office Supplies  205516.0550  41014.5791
        Technology       264973.9810  47462.0351
South   Furniture        117298.6840   6771.2061
        Office Supplies  125651.3130  19986.3928
        Technology       148771.9080  19991.8314
West    Furniture        252612.7435  11504.9503
        Office Supplies  220853.2490  52609.8490
        Technology       251991.8320  44303.6496

In [23]:
df.groupby("Region")["Sales"].agg(["sum","mean","max"]) # Using agg to apply multiple aggregation functions to a column

,sum,mean,max
Region,,,
Central,501239.8908,215.772661,17499.950
East,678781.2400,238.336110,11199.968
South,391721.9050,241.803645,22638.480
West,725457.8245,226.493233,13999.960


In [24]:
subcat_sales = df.groupby(
    ["Region", "Sub-Category"]
)["Sales"].sum()
subcat_sales = subcat_sales.sort_values(ascending=False)
top3 = subcat_sales.groupby(level=0).head(3)
top3

Region   Sub-Category
West     Chairs          101781.328
East     Phones          100614.982
West     Phones           98684.352
East     Chairs           96260.683
Central  Chairs           85230.646
West     Tables           84754.562
Central  Phones           72403.282
East     Storage          71612.584
South    Phones           58304.438
Central  Binders          56923.282
South    Machines         53890.960
         Chairs           45176.446
Name: Sales, dtype: float64

In [25]:
df["Profit Margin %"] = (df["Profit"] / df["Sales"]) * 100

In [26]:
pivot = pd.pivot_table(
    df,
    values="Sales",
    index="Region",
    columns="Segment",
    aggfunc="mean"
)

In [27]:
# Datetime conversion
df["Order Date"] = pd.to_datetime(df["Order Date"]) # Monthly profit
df["Month"] = df["Order Date"].dt.month

monthly_profit = df.groupby("Month")["Profit"].sum()

loss_months = monthly_profit[monthly_profit < 0]

print(loss_months)

Series([], Name: Profit, dtype: float64)


In [28]:
# Quarterly sales
if "Order Date" in df.columns:
    df = df.set_index("Order Date")
elif not isinstance(df.index, pd.DatetimeIndex):
    df.index = pd.to_datetime(df.index)

quarterly_sales = df["Sales"].resample("QE").sum()

print(quarterly_sales)

Order Date
2014-03-31     74447.7960
2014-06-30     86538.7596
2014-09-30    143633.2123
2014-12-31    179627.7302
2015-03-31     68851.7386
2015-06-30     89124.1870
2015-09-30    130259.5752
2015-12-31    182297.0082
2016-03-31     93237.1810
2016-06-30    136082.3010
2016-09-30    143787.3622
2016-12-31    236098.7538
2017-03-31    123144.8602
2017-06-30    133764.3720
2017-09-30    196251.9560
2017-12-31    280054.0670
Freq: QE-DEC, Name: Sales, dtype: float64
